# 12 - Churn Batch Scoring
Loads `retail_lakehouse.ml.churn_model@champion` and scores every customer at the latest snapshot.
Writes `ml.churn_predictions` with a churn probability and risk tier.

Using the **alias** (not a version number) means a new champion is picked up automatically, with no code change.

In [0]:
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
C = "retail_lakehouse"
MODEL_NAME = f"{C}.ml.churn_model"

model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
version = MlflowClient().get_model_version_by_alias(MODEL_NAME, "champion").version
feature_cols = list(model.feature_names_in_)

score_snapshot = spark.sql(f"SELECT MAX(snapshot_date) FROM {C}.ml.customer_features").first()[0]
feats = spark.sql(f"""
  SELECT * FROM {C}.ml.customer_features WHERE snapshot_date = DATE'{score_snapshot}'
""").toPandas()

churn_prob = model.predict_proba(feats[feature_cols])[:, 1]

# Risk tiers by rank: top 20% = High, next 30% = Medium, rest = Low
pct = pd.Series(churn_prob).rank(pct=True)
tier = np.select([pct > 0.80, pct > 0.50], ["High", "Medium"], default="Low")

preds = pd.DataFrame({
    "customer_key": feats["customer_key"],
    "snapshot_date": feats["snapshot_date"],
    "churn_probability": churn_prob.round(4),
    "risk_tier": tier,
    "model_name": MODEL_NAME,
    "model_version": str(version),
})

(spark.createDataFrame(preds)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{C}.ml.churn_predictions"))

spark.sql(f"""COMMENT ON TABLE {C}.ml.churn_predictions IS
  'Churn probability per customer from {MODEL_NAME}@champion. Tiers by rank: High = top 20%, Medium = next 30%.'""")

print(f"✅ Scored {len(preds):,} customers at snapshot {score_snapshot} with model version {version}")

In [0]:
%sql
SELECT p.risk_tier,
       COUNT(*)                              AS customers,
       ROUND(AVG(p.churn_probability), 3)    AS avg_churn_prob,
       ROUND(AVG(f.recency_days), 0)         AS avg_recency_days,
       ROUND(AVG(f.frequency), 2)            AS avg_orders,
       ROUND(AVG(f.monetary), 2)             AS avg_spend,
       ROUND(AVG(f.late_delivery_rate) * 100, 1) AS late_delivery_pct
FROM retail_lakehouse.ml.churn_predictions p
JOIN retail_lakehouse.ml.customer_features f
  ON p.customer_key = f.customer_key AND p.snapshot_date = f.snapshot_date
GROUP BY p.risk_tier
ORDER BY CASE p.risk_tier WHEN 'High' THEN 1 WHEN 'Medium' THEN 2 ELSE 3 END;